In [ ]:
# 1. Install required packages (100% Offline Mode)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 768            # Maximum sequence length
BATCH_SIZE = 16            # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 2  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 1                  # Number of training epochs
LEARNING_RATE = 2e-4        # Learning rate
OUTPUT_DIR = "/kaggle/working/results"  # Output directory on Kaggle

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load NL -> FOL translation datasets, Physics dataset, and Physics Prompt
import os
import json

merged_path = "/kaggle/input/datasets/mduy2911/folc-train/merged_valid.json"
physics_path = "/kaggle/input/datasets/mduy2911/folc-train/physics_distillation.json"
solver_path = "/kaggle/input/datasets/mduy2911/folc-train/solver.md"

def load_translation_dataset(path):
    samples = []
    seen_premises = set()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        samples.append({"premises-NL": nl_list, "premises-FOL": fol_list})
    print(f"Loaded {len(samples)} unique translation samples from {os.path.basename(path)}")
    return samples

def load_physics_dataset(path):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        inp = item.get("input", "")
        out = item.get("output", "")
        if inp and out:
            samples.append({"input": inp, "output": out})
    print(f"Loaded {len(samples)} physics samples from {os.path.basename(path)}")
    return samples

def load_physics_system_prompt(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    print(f"Loaded physics system prompt from {os.path.basename(path)}")
    return content

raw_samples = load_translation_dataset(merged_path)
physics_samples = load_physics_dataset(physics_path)
physics_system_prompt = load_physics_system_prompt(solver_path)


In [ ]:
# 4. Initialize Tokenizer and load Base Model (Standard 16-bit LoRA)
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Using Standard 16-bit LoRA (BF16/FP16) model loading...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    attn_implementation=attn_impl,
    local_files_only=True
)
base_model.config.use_cache = False

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Resume from previous checkpoints if adapter_config.json exists in OUTPUT_DIR
adapter_config_path = os.path.join(OUTPUT_DIR, "adapter_config.json")
if os.path.exists(adapter_config_path):
    print(f"Resuming PEFT adapter weights from {OUTPUT_DIR}...")
    model = PeftModel.from_pretrained(base_model, OUTPUT_DIR, is_trainable=True)
else:
    print("Initializing a new PEFT adapter...")
    model = get_peft_model(base_model, peft_config)

model.print_trainable_parameters()


In [ ]:
# 5. Format Dataset (Chat Template) and split Train/Val
import os
import json
from datasets import Dataset

# Define prompt templates for flat JSON list output with strict count constraint
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n"
    "You must output EXACTLY the same number of formulas as the input premises. Do not skip any premises or merge them.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following {num_premises} premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{premises}\n\n"
    "Return a JSON list of exactly {num_premises} strings containing the formulas, in the exact same order."
)

formatted_samples = []

# Format FOL translation samples (task_id = 0)
for item in raw_samples:
    nl_list = item["premises-NL"]
    fol_list = item["premises-FOL"]
    
    # Format premises as a numbered list
    nl_content = ""
    for i, nl in enumerate(nl_list, start=1):
        nl_content += f"{i}. {nl}\n"
        
    # Render user prompt template with count constraint
    user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
    
    # Assistant response is a flat JSON list of FOL formulas
    assistant_response = json.dumps(fol_list, indent=2)
    
    formatted_samples.append({
        "messages": [
            {"role": "system", "content": system_prompt_template},
            {"role": "user", "content": user_prompt.strip()},
            {"role": "assistant", "content": assistant_response.strip()}
        ],
        "task_id": 0
    })

# Format Physics samples (task_id = 1)
for item in physics_samples:
    physics_input = item["input"]
    physics_output = item["output"]
    
    formatted_samples.append({
        "messages": [
            {"role": "system", "content": physics_system_prompt},
            {"role": "user", "content": physics_input.strip()},
            {"role": "assistant", "content": physics_output.strip()}
        ],
        "task_id": 1
    })

dataset = Dataset.from_list(formatted_samples)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])
dataset = dataset.shuffle(seed=42)
split_dataset = dataset.train_test_split(test_size=0.1)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"Total dataset size: {len(dataset)}")
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")


In [ ]:
# 6. Configure SFTTrainer and start training
import os
import glob
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "fol_loss" in logs:
                print(f"Step {state.global_step}: fol_loss = {logs['fol_loss']:.6f}")
            if "physics_loss" in logs:
                print(f"Step {state.global_step}: physics_loss = {logs['physics_loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")
            if "eval_fol_loss" in logs:
                print(f"Step {state.global_step}: eval_fol_loss = {logs['eval_fol_loss']:.6f}")
            if "eval_physics_loss" in logs:
                print(f"Step {state.global_step}: eval_physics_loss = {logs['eval_physics_loss']:.6f}")

# Custom ReduceLROnPlateau Callback for SFTTrainer
class ReduceLROnPlateauCallback(TrainerCallback):
    def __init__(self, patience=1, factor=0.5, min_lr=1e-6):
        self.patience = patience
        self.factor = factor
        self.min_lr = min_lr
        self.best_loss = float('inf')
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None and "eval_loss" in metrics:
            eval_loss = metrics["eval_loss"]
            if eval_loss < self.best_loss:
                self.best_loss = eval_loss
                self.bad_epochs = 0
            else:
                self.bad_epochs += 1
                if self.bad_epochs >= self.patience:
                    self.bad_epochs = 0
                    optimizer = kwargs.get("optimizer")
                    if optimizer is not None:
                        for param_group in optimizer.param_groups:
                            old_lr = param_group['lr']
                            new_lr = max(old_lr * self.factor, self.min_lr)
                            param_group['lr'] = new_lr
                            print(f"\n[ReduceLROnPlateau] Eval loss did not improve for {self.patience} epoch(s). Reducing learning rate from {old_lr:.2e} to {new_lr:.2e}.\n")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        task_ids = [example.get("task_id", 0) for example in examples]
        batch = super().__call__(examples)
        
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        batch["task_id"] = torch.tensor(task_ids, dtype=torch.long)
        return batch

class CustomSFTTrainer(SFTTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if getattr(self, "_signature_columns", None) is not None:
            if "task_id" not in self._signature_columns:
                self._signature_columns = list(self._signature_columns) + ["task_id"]
            
        self.fol_loss_sum = 0.0
        self.fol_loss_count = 0
        self.physics_loss_sum = 0.0
        self.physics_loss_count = 0
        
        self.eval_fol_loss_sum = 0.0
        self.eval_fol_loss_count = 0
        self.eval_physics_loss_sum = 0.0
        self.eval_physics_loss_count = 0

    def _get_signature_columns(self):
        try:
            columns = super()._get_signature_columns()
        except AttributeError:
            columns = []
        if "task_id" not in columns:
            columns = list(columns) + ["task_id"]
        return columns

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        task_ids = inputs.pop("task_id", None)
        outputs = model(**inputs)
        
        logits = outputs.logits
        labels = inputs.get("labels")
        
        if labels is None or task_ids is None:
            loss = outputs.loss if isinstance(outputs, dict) else outputs[0]
            return (loss, outputs) if return_outputs else loss
            
        shift_logits = logits[..., :-1, :]
        shift_labels = labels[..., 1:]
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        batch_size = shift_logits.size(0)
        
        # Highly optimized, memory-efficient loss calculation to prevent CUDA OOM
        # Only compute cross entropy on active tokens (where label != -100)
        per_token_loss = torch.zeros_like(shift_labels, dtype=logits.dtype)
        active_mask = (shift_labels != -100)
        
        if active_mask.any():
            active_logits = shift_logits[active_mask]
            active_labels = shift_labels[active_mask]
            active_loss = loss_fct(active_logits, active_labels)
            per_token_loss[active_mask] = active_loss.to(logits.dtype)
            
        valid_mask = active_mask.float()
        masked_loss = per_token_loss * valid_mask
        
        sample_losses = masked_loss.sum(dim=-1)
        sample_tokens = valid_mask.sum(dim=-1)
        sample_tokens = torch.clamp(sample_tokens, min=1.0)
        sample_mean_losses = sample_losses / sample_tokens
        
        fol_mask = (task_ids == 0)
        physics_mask = (task_ids == 1)
        
        fol_losses = sample_mean_losses[fol_mask]
        physics_losses = sample_mean_losses[physics_mask]
        
        fol_loss_val = fol_losses.mean() if fol_mask.any() else torch.tensor(0.0, device=logits.device)
        physics_loss_val = physics_losses.mean() if physics_mask.any() else torch.tensor(0.0, device=logits.device)
        
        if model.training:
            if fol_mask.any():
                self.fol_loss_sum += fol_loss_val.detach().item()
                self.fol_loss_count += 1
            if physics_mask.any():
                self.physics_loss_sum += physics_loss_val.detach().item()
                self.physics_loss_count += 1
        else:
            if fol_mask.any():
                self.eval_fol_loss_sum += fol_loss_val.detach().item()
                self.eval_fol_loss_count += 1
            if physics_mask.any():
                self.eval_physics_loss_sum += physics_loss_val.detach().item()
                self.eval_physics_loss_count += 1
                
        total_loss = sample_mean_losses.mean()
        return (total_loss, outputs) if return_outputs else total_loss

    def log(self, logs: Dict[str, float], *args, **kwargs) -> None:
        is_eval = any(k.startswith("eval_") for k in logs.keys())
        if not is_eval:
            if self.fol_loss_count > 0:
                logs["fol_loss"] = self.fol_loss_sum / self.fol_loss_count
                self.fol_loss_sum = 0.0
                self.fol_loss_count = 0
            if self.physics_loss_count > 0:
                logs["physics_loss"] = self.physics_loss_sum / self.physics_loss_count
                self.physics_loss_sum = 0.0
                self.physics_loss_count = 0
        else:
            if self.eval_fol_loss_count > 0:
                logs["eval_fol_loss"] = self.eval_fol_loss_sum / self.eval_fol_loss_count
                self.eval_fol_loss_sum = 0.0
                self.eval_fol_loss_count = 0
            if self.eval_physics_loss_count > 0:
                logs["eval_physics_loss"] = self.eval_physics_loss_sum / self.eval_physics_loss_count
                self.eval_physics_loss_sum = 0.0
                self.eval_physics_loss_count = 0
                
        super().log(logs, *args, **kwargs)

# Mathematically exact, dynamic warmup steps calculation based on actual dataset size
num_samples = len(train_dataset)
effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
steps_per_epoch = num_samples // effective_batch_size
if num_samples % effective_batch_size != 0:
    steps_per_epoch += 1
total_steps = steps_per_epoch * EPOCHS

# Target exactly 3% warmup steps of total training steps (HF v5.2 compliant integer steps)
warmup_steps = max(1, int(total_steps * 0.03))
print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")

# Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
    warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=1,
    logging_first_step=True,
    optim="adamw_torch",
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=BATCH_SIZE,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    neftune_noise_alpha=5.0,
    dataloader_num_workers=2,
    dataloader_pin_memory=True
)

response_template = "<|im_start|>assistant\n"
collator = CustomDataCollator(
    response_template=response_template, 
    tokenizer=tokenizer
)

trainer = CustomSFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config,
    data_collator=collator,
    callbacks=[LossLoggingCallback(), ReduceLROnPlateauCallback(patience=1, factor=0.5)]
)

checkpoints = glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*"))
resume_path = None
if checkpoints:
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    resume_path = checkpoints[-1]
    print(f"Found checkpoints. Resuming training from: {resume_path}")

trainer.train(resume_from_checkpoint=resume_path)

print(f"Saving best validation adapter weights and tokenizer to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Training completed successfully!")


In [ ]:
# 7. Running Inference Test
import gc
import json

# Clean up GPU memory for stable inference
if "trainer" in globals():
    del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()

test_nl_premises = [
    "All humans are mortal.",
    "Socrates is a human."
]

test_nl_content = ""
for i, nl in enumerate(test_nl_premises, start=1):
    test_nl_content += f"{i}. {nl}\n"

# Render test prompt matching new template with count constraint
test_user_prompt = user_prompt_template.format(num_premises=len(test_nl_premises), premises=test_nl_content.strip())

messages = [
    {"role": "system", "content": system_prompt_template},
    {"role": "user", "content": test_user_prompt.strip()}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)

response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- NL Premises ---")
print(test_nl_content.strip())
print("\n--- Model Generated FOL Translation (Expected JSON List) ---")
print(response_text)
print("--------------------------------------------------")

# Test Physics problem solving inference
test_physics_input = (
    "<reasoning_policies></reasoning_policies>\n\n"
    "<question>\n"
    "The measured voltage is 5.00 \u00b1 0.10 V. Calculate the percentage relative error.\n"
    "</question>"
)
physics_messages = [
    {"role": "system", "content": physics_system_prompt},
    {"role": "user", "content": test_physics_input.strip()}
]
prompt_phys = tokenizer.apply_chat_template(physics_messages, tokenize=False, add_generation_prompt=True)
inputs_phys = tokenizer(prompt_phys, return_tensors="pt").to(device)

with torch.no_grad():
    outputs_phys = model.generate(**inputs_phys, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)

response_phys = tokenizer.decode(outputs_phys[0][inputs_phys.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Physics Input ---")
print(test_physics_input.strip())
print("\n--- Model Generated Physics Solution ---")
print(response_phys)
print("--------------------------------------------------")
